---
### part 1: loading, auditing, and cleaning
---
- standardize column names and data types so later code can refer to columns consistently
-raw file is not overwritten. 
- Customer IDs are identifiers, so they are treated as text rather than numbers

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)


OUTPUT_DIR = Path("retail_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
INPUT_FILE = Path("online_retail_II.csv")

RANDOM_SEED = 42

# remove rows that are 100% identical 
REMOVE_EXACT_DUPLICATES = True

In [2]:
df = pd.read_csv(
    INPUT_FILE,
    dtype={
        "Invoice": "string",    # they are all identifiers not measurments so they were converted to string
        "StockCode": "string",
        "Customer ID": "string"
    }
)

df = df.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "price",
    "UnitPrice": "price",
    "Customer ID": "customer_id",
    "CustomerID": "customer_id",
    "Country": "country"
})

# Standardize text columns without changing the raw CSV.
df["stock_code"] = df["stock_code"].str.upper() # 85034A and 85034a
for col in ["invoice", "stock_code", "description", "country", "customer_id"]:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)
    
df["invoice_date"] = pd.to_datetime(    # converts date text into actual datetime values
    df["invoice_date"], errors="coerce"
)

df["quantity"] = pd.to_numeric(
    df["quantity"], errors="coerce"
)

df["price"] = pd.to_numeric(
    df["price"], errors="coerce"
)

# Standardize customer IDs such as "13085.0" to "13085".
df["customer_id"] = df["customer_id"].str.strip()
df["customer_id"] = df["customer_id"].replace("", pd.NA)

df["customer_id"] = df["customer_id"].str.replace(
    r"\.0$", "", regex=True
)

raw_row_count = len(df) # recording original row count

print("Raw row count:", raw_row_count)
print("Dataset shape:", df.shape)
print("First five rows:")
display(df.head())

Raw row count: 1067371
Dataset shape: (1067371, 8)
First five rows:


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


### checking inconsistencies
---
- Find descriptions linked to multiple stock codes and stock codes that appear with a wide range of prices.
- These are review lists, not a list of confirmed mistakes.
- A product may have different descriptions or prices over time for legitimate reasons.


In [3]:
stocks_per_description = (
    df.dropna(subset=["stock_code", "description"])
      .groupby("description")["stock_code"]
      .nunique()
)
inconsistent_descriptions = stocks_per_description[stocks_per_description > 1]
print("Descriptions associated with multiple stock codes:", len(inconsistent_descriptions))
display(
    df[df["description"].isin(inconsistent_descriptions.index)]
      [["description", "stock_code"]]
      .drop_duplicates()
      .sort_values(["description", "stock_code"])
      .head(30)
)

price_summary = (
    df.dropna(subset=["stock_code", "price"])
      .groupby("stock_code")["price"]
      .agg(
          transaction_rows="count",
          unique_prices="nunique",
          minimum_price="min",
          median_price="median",
          maximum_price="max",
      )
)
price_summary["price_range"] = price_summary["maximum_price"] - price_summary["minimum_price"]
display(price_summary.sort_values("price_range", ascending=False).head(20))

Descriptions associated with multiple stock codes: 107


,description,stock_code
428975,?,16235
100428,?,20679
272375,?,20692
817730,?,20694
886199,?,20700
641276,?,20866
754842,?,20867
524457,?,20956
431432,?,21051
768851,?,21147


,transaction_rows,unique_prices,minimum_price,median_price,maximum_price,price_range
stock_code,,,,,,
B,6,5,-53594.360,-24993.965,11062.06,64656.420
M,1426,514,0.000,11.325,38970.00,38970.000
BANK CHARGES,102,58,0.001,42.120,18910.69,18910.689
AMAZONFEE,43,30,1.000,6497.470,17836.46,17835.460
POST,2122,180,0.000,18.000,8142.75,8142.750
ADJUST,67,62,4.570,49.350,5117.03,5112.460
DOT,1446,1291,0.000,154.180,4505.17,4505.170
D,177,162,0.010,19.000,1867.86,1867.850
84016,78,13,0.000,0.420,1157.15,1157.150


---
### dates and transaction ordering
- Check invalid dates, earliest/latest dates, missing customer IDs, and whether rows move backward in time in the CSV.
- Invalid dates cannot be placed reliably in a time split.
- Backward movement in the original row order is not itself an error; the CSV may simply not be sorted chronologically.
- **Keep in mind** Later cells use the actual `invoice_date`, not the row position, to create chronological training, validation, and test periods.

In [4]:
print("Invalid dates:", df["invoice_date"].isna().sum())
print("Earliest date:", df["invoice_date"].min())
print("Latest date:", df["invoice_date"].max())

print(
    "Rows with missing customer IDs:",
    df["customer_id"].isna().sum()
)


Invalid dates: 0
Earliest date: 2009-12-01 07:45:00
Latest date: 2011-12-09 12:50:00
Rows with missing customer IDs: 243007


### Clean and classify transaction records
- **What we are doing:** Remove exact duplicate rows, create flags for invalid fields, identify cancellation invoices, calculate line value, and separate normal purchases from cancellation records.

**Important definitions:**
- `line_total` = `quantity × price` for that transaction line.
- `purchases` contains rows that pass the purchase rules.
- `cancellations` contains **merchandise** cancellations only: invoice starts with `C`, valid date, not a non-product (postage, manual, discount...) and not an entered-then-reversed line. Non-product cancellations are kept in `non_product_cancellations` for reference.
- `is_negative_qty_non_cancel` flags negative quantities on invoices that do NOT start with `C` (stock write-offs / adjustments). They get their own `transaction_type`.
- Customer-specific tables exclude rows without a customer ID.

**Keep in mind:**
- The non-merchandise descriptions/codes are practical rules, not perfect truth.
- A `C` invoice is a cancellation candidate by convention; it does not prove which purchase it reverses.
- **Duplicates are a judgment call.** Exact duplicates may be genuine double scans of the same item. Set `REMOVE_EXACT_DUPLICATES = False` in the settings cell to test results with and without removal.
- Missing customer IDs are not removed from `clean`; they are only excluded from customer-level tables.

In [5]:
import numpy as np
import gc

if REMOVE_EXACT_DUPLICATES:
    clean = df.drop_duplicates().copy()
else:
    clean = df.copy()
duplicates_removed = raw_row_count - len(clean)

clean["invalid_date"] = clean["invoice_date"].isna()
clean["invalid_quantity"] = clean["quantity"].isna() | clean["quantity"].eq(0)
clean["invalid_price"] = clean["price"].isna() | clean["price"].le(0)

# Standard Online Retail II cancellation invoices commonly start with C.
clean["is_cancellation"] = clean["invoice"].str.upper().str.startswith("C", na=False)

# negative quantity on an invoice that does NOT start with C
# (usually stock write-offs / adjustments, not customer sales)
clean["is_negative_qty_non_cancel"] = clean["quantity"].lt(0) & ~clean["is_cancellation"]

description_upper = clean["description"].fillna("").str.upper()
stock_upper = clean["stock_code"].fillna("").str.upper()
non_product_terms = (
    r"POSTAGE|DOTCOM POSTAGE|BANK CHARGES|"
    r"AMAZON FEE|MANUAL|DISCOUNT|ADJUST BAD DEBT|GIFT VOUCHER"
)
known_non_product_codes = {
    "POST", "DOT", "M", "D", "C2", "B", "S",
    "BANK CHARGES", "AMAZONFEE", "ADJUST", "ADJUST2", "CRUK",
    "PADS",     # pads to match all cushions, price 0.001
    "23444",    # "Next Day Carriage", a shipping charge
    "23702",    # "High Resolution Image", a digital file
}

# Gift vouchers (GIFT_0001_10 ...) and test products (TEST001, TEST002 ...)
non_product_code_pattern = r"^(?:GIFT_\d+_\d+|TEST\d+)$"

clean["is_non_product"] = (
    description_upper.str.contains(non_product_terms, regex=True, na=False)
    | stock_upper.isin(known_non_product_codes)
    | stock_upper.str.match(non_product_code_pattern, na=False)
)

clean["line_total"] = clean["quantity"] * clean["price"]

clean["is_purchase"] = (
    ~clean["invalid_date"]
    & ~clean["invalid_quantity"]
    & ~clean["invalid_price"]
    & ~clean["is_cancellation"]
    & ~clean["is_non_product"]
    & clean["quantity"].gt(0)
    & clean["price"].gt(0)
)

clean["transaction_type"] = np.select(
    [
        clean["is_cancellation"],
        clean["invalid_date"],
        clean["is_negative_qty_non_cancel"],
        clean["invalid_price"],
        clean["invalid_quantity"],
        clean["is_non_product"],
        clean["is_purchase"],
    ],
    [
        "cancellation",
        "invalid_date",
        "negative_qty_non_cancel",
        "invalid_price",
        "invalid_quantity",
        "non_product",
        "purchase",
    ],
    default="other",
)

purchases = clean.loc[clean["is_purchase"]].copy()

del df
gc.collect()

39

### Find and exclude lines that were entered and then reversed
- **What we are doing:** Look for a very large purchase line (10,000+ units) that has an exact opposite cancellation line (same customer, same product, same quantity with a minus sign) within 24 hours.
- **Why:** The 80,995-unit line (line total 168,469.60) looks like an order that was typed in by mistake and cancelled minutes later. Left in, it distorts revenue, customer spend and product statistics.
- **Why both sides are removed:** If only the purchase were removed, the cancellation line would still inflate the customer's cancellation value.
- **How to check it:** Read the table this cell prints. If each pair has the same product, the same quantity, opposite signs and a short time gap, the exclusion is justified.
- Only exact round-trips are removed, so a genuinely large wholesale order is never touched.

In [6]:
# Settings for the "entered then reversed" check
REVERSED_MIN_QTY = 10_000     # only very large lines are checked
REVERSED_MAX_HOURS = 24       # the reversal must come within this many hours

line_cols = ["invoice", "customer_id", "stock_code", "invoice_date",
             "quantity", "price", "line_total"]

big_buys = clean.loc[
    ~clean["is_cancellation"]
    & clean["quantity"].ge(REVERSED_MIN_QTY)
    & clean["customer_id"].notna(),
    line_cols,
].rename_axis("buy_row_id").reset_index()

big_cancels = clean.loc[
    clean["is_cancellation"]
    & clean["quantity"].le(-REVERSED_MIN_QTY)
    & clean["customer_id"].notna(),
    line_cols,
].rename_axis("cancel_row_id").reset_index()

pairs = big_buys.merge(
    big_cancels, on=["customer_id", "stock_code"], suffixes=("_buy", "_cancel")
)
pairs["hours_apart"] = (
    (pairs["invoice_date_cancel"] - pairs["invoice_date_buy"]).dt.total_seconds() / 3600
)
pairs = pairs.loc[
    pairs["quantity_buy"].eq(-pairs["quantity_cancel"])
    & pairs["hours_apart"].between(0, REVERSED_MAX_HOURS)
].copy()

print("Entered-then-reversed pairs found:", len(pairs))
display(pairs[[
    "customer_id", "stock_code", "invoice_buy", "invoice_cancel",
    "quantity_buy", "quantity_cancel", "price_buy", "line_total_buy",
    "invoice_date_buy", "invoice_date_cancel", "hours_apart",
]])

clean["is_reversed_entry"] = False
clean.loc[pairs["buy_row_id"].tolist(), "is_reversed_entry"] = True
clean.loc[pairs["cancel_row_id"].tolist(), "is_reversed_entry"] = True

clean["is_purchase"] = clean["is_purchase"] & ~clean["is_reversed_entry"]
clean.loc[clean["is_reversed_entry"], "transaction_type"] = "reversed_entry"
purchases = clean.loc[clean["is_purchase"]].copy()

print("Rows excluded as reversed entries:", int(clean["is_reversed_entry"].sum()))
print("Largest purchase quantity now:", purchases["quantity"].max())
print("Largest purchase line total now:", round(purchases["line_total"].max(), 2))

Entered-then-reversed pairs found: 2


,customer_id,stock_code,invoice_buy,invoice_cancel,quantity_buy,quantity_cancel,price_buy,line_total_buy,invoice_date_buy,invoice_date_cancel,hours_apart
0,12346,23166,541431,C541433,74215,-74215,1.04,77183.6,2011-01-18 10:01:00,2011-01-18 10:17:00,0.266667
1,16446,23843,581483,C581484,80995,-80995,2.08,168469.6,2011-12-09 09:15:00,2011-12-09 09:27:00,0.200000


Rows excluded as reversed entries: 4
Largest purchase quantity now: 19152
Largest purchase line total now: 38970.0


- what did the rule exclude?

In [7]:
excluded = clean.loc[clean["is_non_product"]].copy()
excluded["matched_by"] = np.where(
    stock_upper[excluded.index].isin(known_non_product_codes)
    | stock_upper[excluded.index].str.match(non_product_code_pattern, na=False),
    "stock code list",
    "description words",
)
excluded["matched_word"] = description_upper[excluded.index].str.extract(
    "(" + non_product_terms + ")"
)[0]

excluded_audit = (
    excluded.groupby(["stock_code", "description", "matched_by", "matched_word"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)
display(excluded_audit.head(40))

,stock_code,description,matched_by,matched_word,rows
37,POST,POSTAGE,stock code list,POSTAGE,2079
16,DOT,DOTCOM POSTAGE,stock code list,DOTCOM POSTAGE,1423
35,M,Manual,stock code list,MANUAL,1392
12,C2,CARRIAGE,stock code list,<NA>,274
15,D,Discount,stock code list,DISCOUNT,173
39,S,SAMPLES,stock code list,<NA>,101
11,BANK CHARGES,Bank Charges,stock code list,BANK CHARGES,100
1,23444,Next Day Carriage,stock code list,<NA>,80
6,ADJUST,Adjustment by john on 26/01/2010 16,stock code list,<NA>,38
9,AMAZONFEE,AMAZON FEE,stock code list,AMAZON FEE,36


- false exclusions, the most suspicious ones

In [8]:
real_code_pattern = r"^\d{5}[A-Z]{0,2}$"   # real products look like 85123A

wrongly_excluded = excluded.loc[
    excluded["stock_code"].str.upper().str.match(real_code_pattern, na=False)
]
display(
    wrongly_excluded.groupby(["stock_code", "description", "matched_word"])
    .size().reset_index(name="rows").sort_values("rows", ascending=False).head(30)
)

,stock_code,description,matched_word,rows
0,22016,Dotcomgiftshop Gift Voucher £100.00,GIFT VOUCHER,1


false inclusions, things kept that may not be products

In [9]:
code_looks_real = purchases["stock_code"].str.match(real_code_pattern, na=False)

# Check 1: kept lines whose code does not look like a normal product code
print("CHECK 1: odd codes that were kept")
check1 = (
    purchases.loc[~code_looks_real]
    .groupby(["stock_code", "description"], dropna=False)
    .agg(rows=("invoice", "size"), median_price=("price", "median"))
    .reset_index()
    .sort_values("rows", ascending=False)
)
display(check1.head(50))

# Check 2: normal-looking code, but the description has small letters.
# Real products are almost always written in CAPITALS; staff notes are not.
print("CHECK 2: descriptions with lowercase letters")
has_lowercase = purchases["description"].fillna("").str.contains(r"[a-z]")
check2 = (
    purchases.loc[has_lowercase]
    .groupby(["stock_code", "description"], dropna=False)
    .agg(rows=("invoice", "size"), median_price=("price", "median"))
    .reset_index()
    .sort_values("rows", ascending=False)
)
display(check2.head(40))

# Check 3: kept lines with a tiny price (real sales are rarely under 5 pence)
print("CHECK 3: very cheap items")
check3 = (
    purchases.loc[purchases["price"] < 0.05]
    .groupby(["stock_code", "description"], dropna=False)
    .agg(rows=("invoice", "size"), median_price=("price", "median"))
    .reset_index()
    .sort_values("rows", ascending=False)
)
display(check3.head(30))

print("CHECK 4: very expensive items")
check4 = (
    purchases.loc[purchases["price"] > 200]
    .groupby(["stock_code", "description"], dropna=False)
    .agg(rows=("invoice", "size"), median_price=("price", "median"))
    .reset_index()
    .sort_values("median_price", ascending=False)
)
display(check4.head(20))

CHECK 1: odd codes that were kept


,stock_code,description,rows,median_price
5,DCGS0058,MISO PRETTY GUM,30,0.83
15,DCGSSGIRL,GIRLS PARTY BAG,23,3.29
14,DCGSSBOY,BOYS PARTY BAG,21,3.29
13,DCGS0076,SUNJAR LED NIGHT NIGHT LIGHT,14,16.13
0,DCGS0003,BOXED GLASS ASHTRAY,13,2.51
9,DCGS0069,OOH LA LA DOGS COLLAR,5,16.13
1,DCGS0004,HAYNES CAMPER SHOULDER BAG,4,16.98
11,DCGS0072,CAT CAMOUFLAGUE COLLAR,3,5.17
7,DCGS0066N,NAVY CUDDLES DOG HOODIE,2,8.65
8,DCGS0068,DOGS NIGHT COLLAR,2,8.56


CHECK 2: descriptions with lowercase letters


,stock_code,description,rows,median_price
11,21703,BAG 125g SWIRLY MARBLES,577,0.42
12,21704,BAG 250g SWIRLY MARBLES,574,0.85
26,46000S,POLYESTER FILLER PAD 40x40cm,380,1.45
23,46000M,POLYESTER FILLER PAD 45x45cm,354,1.55
22,22965,3 TRADITIONAl BISCUIT CUTTERS SET,252,2.10
13,21705,BAG 500g SWIRLY MARBLES,228,1.65
16,22686,FRENCH BLUE METAL DOOR SIGN No,173,2.46
2,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,74,0.18
25,46000R,POLYESTER FILLER PAD 45x30cm,61,1.45
27,46000U,POLYESTER FILLER PAD 30CMx30CM,51,1.25


CHECK 3: very cheap items


,stock_code,description,rows,median_price
0,16045,POPART WOODEN PENCILS ASST,66,0.04
1,16053,POPART COL BALLPOINT PEN ASST,2,0.03
2,17061,ASSORTED SHAPED STENCIL FOR HENNA,1,0.04


CHECK 4: very expensive items


,stock_code,description,rows,median_price
0,22502,PICNIC BASKET WICKER 60 PIECES,2,649.5
3,84016,FLAG OF ST GEORGE CAR FLAG,5,408.4
1,22655,VINTAGE RED KITCHEN CABINET,20,295.0
2,22656,VINTAGE BLUE KITCHEN CABINET,20,295.0


In [10]:
purchases = clean.loc[clean["is_purchase"]].copy()

# Only real-product cancellations are used for modelling.
# Non-product cancellations (postage, manual, discount...) and reversed entries are kept apart.
cancellations = clean.loc[
    clean["is_cancellation"]
    & ~clean["invalid_date"]
    & ~clean["is_non_product"]
    & ~clean["is_reversed_entry"]
].copy()
non_product_cancellations = clean.loc[
    clean["is_cancellation"]
    & ~clean["invalid_date"]
    & clean["is_non_product"]
].copy()

customer_purchases = purchases.dropna(subset=["customer_id"]).copy()
customer_cancellations = cancellations.dropna(subset=["customer_id"]).copy()

cleaning_summary = pd.DataFrame([
    {"Metric": "Raw rows", "Count": raw_row_count},
    {"Metric": "Exact duplicates removed", "Count": duplicates_removed},
    {"Metric": "Rows after exact deduplication", "Count": len(clean)},
    {"Metric": "Invalid-date rows retained in combined table", "Count": int(clean["invalid_date"].sum())},
    {"Metric": "Valid merchandise purchase rows", "Count": len(purchases)},
    {"Metric": "Merchandise cancellation rows with valid dates", "Count": len(cancellations)},
    {"Metric": "Non-product cancellation rows (kept apart, not modelled)", "Count": len(non_product_cancellations)},
    {"Metric": "Entered-then-reversed rows excluded (both sides)", "Count": int(clean["is_reversed_entry"].sum())},
    {"Metric": "Rows with missing customer IDs", "Count": int(clean["customer_id"].isna().sum())},
    {"Metric": "Rows with non-positive or missing price", "Count": int(clean["invalid_price"].sum())},
    {"Metric": "Rows with zero or missing quantity", "Count": int(clean["invalid_quantity"].sum())},
    {"Metric": "Negative-quantity rows on non-C invoices", "Count": int(clean["is_negative_qty_non_cancel"].sum())},
    {"Metric": "Flagged non-merchandise rows", "Count": int(clean["is_non_product"].sum())},
])
display(cleaning_summary)
print("Purchases:", len(purchases))
print("Cancellations (merchandise only):", len(cancellations))

print("\nHow every row was classified:")
display(clean["transaction_type"].value_counts())

print("Most common descriptions on negative-quantity rows of non-C invoices:")
display(
    clean.loc[clean["is_negative_qty_non_cancel"], "description"]
    .value_counts(dropna=False).head(10)
)

,Metric,Count
0,Raw rows,1067371
1,Exact duplicates removed,34335
2,Rows after exact deduplication,1033036
3,Invalid-date rows retained in combined table,0
4,Valid merchandise purchase rows,1003255
5,Merchandise cancellation rows with valid dates,17911
6,"Non-product cancellation rows (kept apart, not...",1191
7,Entered-then-reversed rows excluded (both sides),4
8,Rows with missing customer IDs,235151
9,Rows with non-positive or missing price,6019


Purchases: 1003255
Cancellations (merchandise only): 17911

How every row was classified:


transaction_type
purchase                   1003255
cancellation                 19102
non_product                   4656
negative_qty_non_cancel       3393
invalid_price                 2626
reversed_entry                   4
Name: count, dtype: int64

Most common descriptions on negative-quantity rows of non-C invoices:


description
<NA>                     2633
check                     121
damages                    83
?                          81
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Name: count, dtype: int64[pyarrow]

### Flag unusual numeric values
- **What we are doing:** Use the IQR rule to flag unusually low or high quantities, prices, and line totals in the purchase table.
- **What to look for:** The output gives quartiles, flagging boundaries, and the number/percentage outside those boundaries.
- **Keep in mind:** An outlier is not automatically bad data. Wholesale purchases can legitimately have large quantities. This cell only reports unusual values; it does not delete or cap them.

In [11]:
def iqr_outlier_summary(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return {
            "count": 0, "q1": np.nan, "q3": np.nan, "lower_bound": np.nan,
            "upper_bound": np.nan, "outlier_count": 0, "outlier_percentage": np.nan
        }
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged = (values < lower) | (values > upper)
    return {
        "count": len(values), "q1": q1, "q3": q3,
        "lower_bound": lower, "upper_bound": upper,
        "outlier_count": int(flagged.sum()),
        "outlier_percentage": round(flagged.mean() * 100, 2),
    }

outlier_summary = pd.DataFrame({
    col: iqr_outlier_summary(purchases[col])
    for col in ["quantity", "price", "line_total"]
}).T
display(outlier_summary)

,count,q1,q3,lower_bound,upper_bound,outlier_count,outlier_percentage
quantity,1003255.0,1.00,12.00,-15.500,28.500,53017.0,5.28
price,1003255.0,1.25,4.13,-3.070,8.450,74983.0,7.47
line_total,1003255.0,4.13,17.70,-16.225,38.055,82133.0,8.19


### Define chronological purchase splits
- **What we are doing:** Divide purchase transactions by date into training (through March 31, 2011), validation (April–June 2011), and test (after June 30, 2011).
- **Why dates matter:** Models learn from the past and are evaluated on later transactions, which better reflects how recommendations would work in practice.
- **Keep in mind:** Do not randomly shuffle transactions across these splits.
- Check the displayed date ranges and row counts. If a split is empty, the code stops because later evaluation would not be meaningful.

In [12]:
TRAIN_END = pd.Timestamp("2011-03-31 23:59:59")
VALIDATION_END = pd.Timestamp("2011-06-30 23:59:59")

train_transactions = purchases.loc[
    purchases["invoice_date"] <= TRAIN_END
].copy()
validation_transactions = purchases.loc[
    (purchases["invoice_date"] > TRAIN_END)
    & (purchases["invoice_date"] <= VALIDATION_END)
].copy()
test_transactions = purchases.loc[
    purchases["invoice_date"] > VALIDATION_END
].copy()

split_parts = {
    "train": train_transactions,
    "validation": validation_transactions,
    "test": test_transactions,
}
empty_splits = [name for name, part in split_parts.items() if part.empty]
if empty_splits:
    raise ValueError(
        f"Empty purchase split(s): {empty_splits}. "
        "Check the source date range and split boundaries."
    )

split_summary = pd.DataFrame([
    {
        "Split": name.title(),
        "Start Date": part["invoice_date"].min(),
        "End Date": part["invoice_date"].max(),
        "Rows": len(part),
        "Customers": part["customer_id"].nunique(),
        "Invoices": part["invoice"].nunique(),
    }
    for name, part in split_parts.items()
])
display(split_summary)

,Split,Start Date,End Date,Rows,Customers,Invoices
0,Train,2009-12-01 07:45:00,2011-03-31 19:55:00,617572,4691,24906
1,Validation,2011-04-01 08:22:00,2011-06-30 20:08:00,99976,1988,4428
2,Test,2011-07-01 08:16:00,2011-12-09 12:50:00,285707,3359,10177


### Create a standardized training product catalog
- **What we are doing:** Build one row per product code using only training-period purchases. 
- It includes typical prices, quantities, number of lines/orders, revenue and a representative description.
- **How the description is chosen:** Descriptions are uppercased/trimmed, then the most frequently observed description for each stock code is selected.
- **Keep in mind:** This is a practical standardized label, not proof that every description variation is the same real-world product. 
- Using training data only prevents later sales from influencing training-period catalog statistics.

In [13]:
# `train_transactions` comes from `purchases`, which already excludes non-products
train_catalog_source = train_transactions.loc[
    train_transactions["description"].notna()
    & train_transactions["description"].ne("")
].copy()

# Use a cleaned description for grouping while preserving the original column.
train_catalog_source["description_clean"] = (
    train_catalog_source["description"].str.upper().str.strip()
)

description_counts = (
    train_catalog_source.groupby(["stock_code", "description_clean"])
    .size()
    .reset_index(name="count")
    .sort_values(
        ["stock_code", "count", "description_clean"],
        ascending=[True, False, True],
    )
)
standard_descriptions = (
    description_counts.drop_duplicates("stock_code")
    [["stock_code", "description_clean"]]
    .rename(columns={"description_clean": "standard_description"})
)

train_product_catalog = (
    train_catalog_source.groupby("stock_code")
    .agg(
        average_price=("price", "mean"),
        median_price=("price", "median"),
        total_quantity=("quantity", "sum"),
        purchase_lines=("invoice", "size"),
        order_count=("invoice", "nunique"),
        total_revenue=("line_total", "sum"),
    )
    .reset_index()
    .merge(standard_descriptions, on="stock_code", how="left")
)
display(train_product_catalog.head())

,stock_code,average_price,median_price,total_quantity,purchase_lines,order_count,total_revenue,standard_description
0,10002,0.980939,0.85,8482,362,357,6781.61,INFLATABLE POLITICAL GLOBE
1,10002R,5.133333,5.17,4,3,3,20.57,ROBOT PENCIL SHARPNER
2,10080,0.850000,0.85,14,7,6,11.90,GROOVY CACTUS INFLATABLE
3,10109,0.420000,0.42,4,1,1,1.68,BENDY COLOUR PENCILS
4,10120,0.251509,0.21,546,53,53,117.96,DOGGY RUBBER


### Create training customer-product interactions
- **What we are doing:** Summarize each known customer's purchases of each product during the training period. 
- We also create integer mappings for customers/products for algorithms that need numeric indices.
- **What the table means:** One row represents a customer–product pair, not one transaction. It summarizes quantity, purchase lines, orders, last purchase date, and value.
- **Critical caution:** Fit collaborative filtering from this training interaction table only. 
Do not build the training matrix from validation/test purchases, or the model would get information from the future. 
- Customers/products first seen later may not have a training index; handle them as cold-start cases.

In [14]:
train_interaction_source = train_transactions.dropna(
    subset=["customer_id"]
).copy()


train_customer_product = (
    train_interaction_source.groupby(["customer_id", "stock_code"])
    .agg(
        quantity=("quantity", "sum"),
        number_of_purchase_lines=("invoice", "size"),
        number_of_orders=("invoice", "nunique"),
        last_purchase_date=("invoice_date", "max"),
        total_value=("line_total", "sum"),
    )
    .reset_index()
)

train_customer_mapping = pd.DataFrame({
    "customer_id": sorted(train_customer_product["customer_id"].unique())
})
train_customer_mapping["customer_index"] = np.arange(len(train_customer_mapping))

train_product_mapping = pd.DataFrame({
    "stock_code": sorted(train_customer_product["stock_code"].unique())
})
train_product_mapping["product_index"] = np.arange(len(train_product_mapping))

print("Training interaction rows:", len(train_customer_product))
print("Training customers:", train_customer_product["customer_id"].nunique())
print("Training products:", train_customer_product["stock_code"].nunique())

Training interaction rows: 317018
Training customers: 4691
Training products: 4089


### Match each cancellation to the purchases it most likely reverses (oldest-first)
- **What we are doing:** For every customer-linked merchandise cancellation, look at that customer's earlier purchases of the same product, no older than `MAX_MATCH_DAYS` (180) days, and use them **oldest-first**.
- **What "oldest-first" means:** Each purchase has a number of units that can still be "cancelled". A cancellation takes units from the oldest eligible purchase first, then the next oldest, until its quantity is covered. Used-up units cannot be used again.
- **Why this is tighter than before:** One purchase can no longer be "cancelled" more times than it was bought, and a cancellation is no longer left with several equally likely candidates.
- **Match statuses:**
  - `matched_single_invoice`: fully covered by one purchase invoice.
  - `matched_multiple_invoices`: fully covered, but by units from 2 or more invoices.
  - `partial_match`: only part of the quantity could be covered (the rest has no earlier purchase left).
  - `unmatched`: nothing could be matched; `unmatched_reason` says why.
- **Critical caution:** Even a full match is only plausible, not proof. The dataset never records which purchase a cancellation reverses.
- Setting `MAX_MATCH_DAYS = None` removes the age limit.

In [15]:
# How far back a purchase may be to count as a possible match for a cancellation.
# Set to None to allow purchases of any age.
MAX_MATCH_DAYS = 180

# Both tables are already "merchandise only" (see the cleaning summary cell).
cancellation_lookup = customer_cancellations.copy()

keep_cols = ["invoice", "customer_id", "stock_code",
             "description", "invoice_date", "quantity"]

# Only the (customer, product) pairs that have a cancellation
cancel_pairs = cancellation_lookup[["customer_id", "stock_code"]].drop_duplicates()

purchase_lookup = (
    customer_purchases[keep_cols]
    .merge(cancel_pairs, on=["customer_id", "stock_code"], how="inner")
    .sort_values("invoice_date", kind="stable")      # oldest purchase first
)

purchase_groups = {
    key: group
    for key, group in purchase_lookup.groupby(["customer_id", "stock_code"])
}
empty_purchases = purchase_lookup.iloc[0:0]
gc.collect()

def split_name(date):
    if date <= TRAIN_END:
        return "train"
    if date <= VALIDATION_END:
        return "validation"
    return "test"

link_results = []

# Work through each (customer, product) pair separately, cancellations in date order.
cancel_sorted = cancellation_lookup.sort_values("invoice_date", kind="stable")

for key, cancels in cancel_sorted.groupby(["customer_id", "stock_code"]):
    group = purchase_groups.get(key)
    if group is None:
        dates = np.array([], dtype="datetime64[ns]")
        remaining = np.array([], dtype=float)
        invoices = np.array([], dtype=object)
    else:
        dates = group["invoice_date"].to_numpy()
        remaining = group["quantity"].to_numpy().astype(float)  # units not yet "used up"
        invoices = group["invoice"].to_numpy()

    for cancel in cancels.itertuples():
        cancel_date = cancel.invoice_date
        cancel_np = cancel_date.to_datetime64()
        need = float(abs(cancel.quantity))

        is_earlier = dates < cancel_np
        if MAX_MATCH_DAYS is None:
            eligible = is_earlier
        else:
            eligible = is_earlier & (dates >= cancel_np - np.timedelta64(MAX_MATCH_DAYS, "D"))

        # Oldest-first allocation: take units from the oldest eligible purchase,
        # then the next oldest, until the cancelled quantity is covered.
        left = need
        used_invoices, used_dates = [], []
        for pos in np.flatnonzero(eligible):          # already in oldest-first order
            if left <= 0:
                break
            if remaining[pos] <= 0:
                continue
            take = min(remaining[pos], left)
            remaining[pos] -= take
            left -= take
            used_invoices.append(invoices[pos])
            used_dates.append(dates[pos])

        matched = need - left
        distinct_invoices = list(dict.fromkeys(used_invoices))   # unique, order kept

        if matched == 0:
            status = "unmatched"
            confidence = "unmatched"
            if group is None:
                reason = "customer never bought this product"
            elif not is_earlier.any():
                reason = "bought only after the cancellation"
            elif not eligible.any():
                reason = "earlier purchases exist but all are older than the window"
            else:
                reason = "earlier purchases were already used up by other cancellations"
        else:
            reason = ""
            if left > 0:
                status = "partial_match"
                confidence = "weak"
            elif len(distinct_invoices) == 1:
                status = "matched_single_invoice"
                confidence = "plausible_unconfirmed"
            else:
                status = "matched_multiple_invoices"
                confidence = "plausible_unconfirmed"

        link_results.append({
            "cancellation_row_id": cancel.Index,
            "cancellation_invoice": cancel.invoice,
            "customer_id": cancel.customer_id,
            "stock_code": cancel.stock_code,
            "cancellation_date": cancel_date,
            "split": split_name(cancel_date),
            "cancellation_quantity": cancel.quantity,
            "matched_quantity": matched,
            "unmatched_quantity": left,
            "matched_invoice_count": len(distinct_invoices),
            "first_matched_invoice": distinct_invoices[0] if distinct_invoices else pd.NA,
            "matched_invoices": ";".join(distinct_invoices),
            "days_since_oldest_match": (
                (cancel_date - pd.Timestamp(used_dates[0])).days if used_dates else np.nan
            ),
            "days_since_newest_match": (
                (cancel_date - pd.Timestamp(used_dates[-1])).days if used_dates else np.nan
            ),
            "match_status": status,
            "link_confidence": confidence,   # nothing here can be "confirmed"
            "unmatched_reason": reason,
        })

cancellation_links = pd.DataFrame(link_results)
display(cancellation_links["match_status"].value_counts())

match_status
matched_single_invoice       14616
unmatched                     2142
matched_multiple_invoices      687
partial_match                  138
Name: count, dtype: int64

In [16]:
# Step 1: the summary
display(pd.crosstab(cancellation_links["split"], cancellation_links["match_status"], margins=True))
display(cancellation_links.loc[cancellation_links["match_status"] == "unmatched",
                               "unmatched_reason"].value_counts())
print("Confirmed by source data: 0 (the dataset has no cancellation-to-purchase link)")

# Step 2: look at a few real examples of each kind
def show_case(cancel_row_id):
    row = cancellation_links.loc[cancellation_links["cancellation_row_id"] == cancel_row_id].iloc[0]
    cancel = cancellation_lookup.loc[[cancel_row_id],
        ["invoice", "customer_id", "stock_code", "description", "invoice_date", "quantity"]]
    print("CANCELLATION:")
    display(cancel)
    key = (cancel["customer_id"].iloc[0], cancel["stock_code"].iloc[0])
    earlier = purchase_groups.get(key, empty_purchases)
    earlier = earlier.loc[earlier["invoice_date"] < cancel["invoice_date"].iloc[0],
        ["invoice", "customer_id", "stock_code", "description", "invoice_date", "quantity"]].copy()
    earlier["used_by_this_cancellation"] = earlier["invoice"].isin(row["matched_invoices"].split(";"))
    print("EARLIER PURCHASES OF THE SAME PRODUCT BY THE SAME CUSTOMER:")
    display(earlier)

for status in ["matched_single_invoice", "matched_multiple_invoices", "partial_match", "unmatched"]:
    pool = cancellation_links.loc[cancellation_links["match_status"] == status, "cancellation_row_id"]
    print("=" * 20, status, f"({len(pool)} rows)")
    if pool.empty:
        continue
    for row_id in pool.sample(min(3, len(pool)), random_state=RANDOM_SEED):
        show_case(row_id)

# Step 3: sanity checks (these should all be 0, except last 1 which is just information)
matched_rows = cancellation_links.loc[cancellation_links["matched_quantity"] > 0]
print("Matched purchase dated after the cancellation:",
      int((matched_rows["days_since_newest_match"] < 0).sum()))
print("Matched quantity bigger than the cancelled quantity:",
      int((cancellation_links["matched_quantity"]
           > cancellation_links["cancellation_quantity"].abs()).sum()))
if MAX_MATCH_DAYS is not None:
    print(f"Matched purchase older than {MAX_MATCH_DAYS} days:",
          int((matched_rows["days_since_oldest_match"] > MAX_MATCH_DAYS).sum()))
print("Cancellation rows with more than one matched invoice:",
      int((cancellation_links["matched_invoice_count"] > 1).sum()))

match_status,matched_multiple_invoices,matched_single_invoice,partial_match,unmatched,All
split,,,,,
test,176,3816,40,413,4445
train,457,9226,91,1655,11429
validation,54,1574,7,74,1709
All,687,14616,138,2142,17583


unmatched_reason
customer never bought this product                               1034
earlier purchases exist but all are older than the window         555
bought only after the cancellation                                498
earlier purchases were already used up by other cancellations      55
Name: count, dtype: int64

Confirmed by source data: 0 (the dataset has no cancellation-to-purchase link)
==================== matched_single_invoice (14616 rows)
CANCELLATION:


,invoice,customer_id,stock_code,description,invoice_date,quantity
374718,C525536,15093,22467,GUMBALL COAT RACK,2010-10-05 19:09:00,-25


EARLIER PURCHASES OF THE SAME PRODUCT BY THE SAME CUSTOMER:


...

CANCELLATION:


,invoice,customer_id,stock_code,description,invoice_date,quantity
9157,C490120,14277,21080,SET/20 RED SPOTTY PAPER NAPKINS,2009-12-03 17:52:00,-12


EARLIER PURCHASES OF THE SAME PRODUCT BY THE SAME CUSTOMER:


,invoice,customer_id,stock_code,description,invoice_date,quantity,used_by_this_cancellation


Matched purchase dated after the cancellation: 0
Matched quantity bigger than the cancelled quantity: 0
Matched purchase older than 180 days: 0
Cancellation rows with more than one matched invoice: 698


### Build training-period cancellation indicators
- **What we are doing:** Calculate customer-level and product-level cancellation summaries from training-period transactions only. These are potential inputs for cancellation-aware ranking or descriptive dashboard views.
- **How to read the rates:** They compare observed cancellation invoices with the combined count of purchase and cancellation invoices for that customer/product.
- **Critical caution:** These are proxy rates, not verified probabilities that a purchased item will be returned. Invoice conventions and imperfect matching affect them. 
- Small counts can also make a rate unstable. Do not interpret a high rate as causal evidence that a product is bad.
- **Why training only:** Using later cancellation activity would leak future information into a ranking/model that is supposed to operate earlier.

In [17]:
# `cancellations` already contains merchandise cancellations only
train_cancellations = cancellations.loc[
    cancellations["invoice_date"] <= TRAIN_END
].copy()

train_known_cancellations = train_cancellations.dropna(
    subset=["customer_id"]
).copy()

customer_purchase_counts = (
    train_transactions.dropna(subset=["customer_id"])
    .groupby("customer_id")["invoice"]
    .nunique()
    .rename("purchase_invoice_count")
)
customer_cancel_counts = (
    train_known_cancellations.groupby("customer_id")["invoice"]
    .nunique()
    .rename("cancellation_invoice_count")
)
customer_cancel_rates = pd.concat(
    [customer_purchase_counts, customer_cancel_counts], axis=1
).fillna(0)
customer_cancel_rates["cancellation_rate"] = (
    customer_cancel_rates["cancellation_invoice_count"]
    / (
        customer_cancel_rates["purchase_invoice_count"]
        + customer_cancel_rates["cancellation_invoice_count"]
    ).replace(0, np.nan)
).fillna(0)
customer_cancel_rates = customer_cancel_rates.reset_index()

product_purchase_counts = (
    train_transactions.groupby("stock_code")["invoice"]
    .nunique()
    .rename("purchase_invoice_count")
)
product_cancel_counts = (
    train_cancellations.groupby("stock_code")["invoice"]
    .nunique()
    .rename("cancellation_invoice_count")
)
product_cancel_rates = pd.concat(
    [product_purchase_counts, product_cancel_counts], axis=1
).fillna(0)
product_cancel_rates["cancellation_rate"] = (
    product_cancel_rates["cancellation_invoice_count"]
    / (
        product_cancel_rates["purchase_invoice_count"]
        + product_cancel_rates["cancellation_invoice_count"]
    ).replace(0, np.nan)
).fillna(0)
product_cancel_rates = product_cancel_rates.reset_index()

display(customer_cancel_rates.head())
display(product_cancel_rates.head())

,customer_id,purchase_invoice_count,cancellation_invoice_count,cancellation_rate
0,12346,2.0,0.0,0.000000
1,12347,3.0,0.0,0.000000
2,12348,3.0,0.0,0.000000
3,12349,2.0,1.0,0.333333
4,12350,1.0,0.0,0.000000


,stock_code,purchase_invoice_count,cancellation_invoice_count,cancellation_rate
0,10002,357.0,7.0,0.019231
1,10002R,3.0,0.0,0.000000
2,10080,6.0,0.0,0.000000
3,10109,1.0,0.0,0.000000
4,10120,53.0,1.0,0.018519


### Build customer snapshots and lapse labels
- **This is one of the most important cells.** It creates training examples for the customer-lapse model. 
- Each row represents one customer as of a particular snapshot date—not a customer summarized using all future data.
#### **Features vs label:**
- Features use purchases before the snapshot date (past behavior).
- `lapse = 1` means no qualifying purchase in the next 90 days.
- `lapse = 0` means at least one qualifying purchase in that 90-day window.
 - A snapshot is skipped if the dataset does not extend far enough to observe all
   90 future days. Without that full window, we cannot fairly assign the label.
- **Why this matters:** If future purchases were included in the features, the model would indirectly see the answer it is meant to predict (data leakage).
- **Keep in mind:** The label means “no purchase during this defined window,” not that the customer permanently left the business. 
- A customer may return later.
- The snapshot dates and 90-day window are project choices that should be documented.

In [18]:
LABEL_WINDOW_DAYS = 90

# Each split's 90-day label window must END before the next split's snapshot starts.
# (Mar 1 + 90 days = May 30 <= Jun 1;  Jun 1 + 90 days = Aug 30 <= Sep 1)
TRAIN_SNAPSHOT_DATES = list(pd.to_datetime(
    ["2010-06-01", "2010-09-01", "2010-12-01", "2011-03-01"]
))
VALIDATION_SNAPSHOT_DATE = pd.Timestamp("2011-06-01")
TEST_SNAPSHOT_DATE = pd.Timestamp("2011-09-01")
SNAPSHOT_DATES = TRAIN_SNAPSHOT_DATES + [VALIDATION_SNAPSHOT_DATE, TEST_SNAPSHOT_DATE]

# purchases / cancellations are already merchandise-only, so no extra filtering is needed
purchase_data = customer_purchases.copy()
cancellation_data = customer_cancellations.copy()

data_end_date = clean["invoice_date"].max()
if pd.isna(data_end_date) or purchase_data.empty:
    raise ValueError("No valid transaction dates or customer-known purchases are available.")

# keep only the columns we need (smaller = faster)
purchase_data = purchase_data[["customer_id", "invoice", "stock_code", "invoice_date",
                               "quantity", "line_total", "country"]].sort_values(
                               "invoice_date", kind="stable")
cancellation_data = cancellation_data[["customer_id", "invoice", "invoice_date", "line_total"]]

def per_customer(frame, col, how, name, index):
    return frame.groupby("customer_id")[col].agg(how).reindex(index).fillna(0).rename(name)

snapshot_tables = []
for snapshot_date in SNAPSHOT_DATES:
    label_end = snapshot_date + pd.Timedelta(days=LABEL_WINDOW_DAYS)
    if label_end > data_end_date:
        print(f"Skipping {snapshot_date.date()}: incomplete 90-day label window.")
        continue

    hist = purchase_data.loc[purchase_data["invoice_date"] < snapshot_date]
    if hist.empty:
        continue
    future_buyers = purchase_data.loc[
        (purchase_data["invoice_date"] > snapshot_date)
        & (purchase_data["invoice_date"] <= label_end), "customer_id"].unique()
    canc = cancellation_data.loc[cancellation_data["invoice_date"] < snapshot_date]

    g = hist.groupby("customer_id")
    out = pd.DataFrame({
        "last_date": g["invoice_date"].max(),
        "first_date": g["invoice_date"].min(),
        "frequency_orders": g["invoice"].nunique(),
        "monetary": g["line_total"].sum(),
        "total_quantity": g["quantity"].sum(),
        "distinct_products": g["stock_code"].nunique(),
        "country": g["country"].last(),
    })
    idx = out.index

    orders = (hist.groupby(["customer_id", "invoice"])["invoice_date"].min()
                  .reset_index().sort_values(["customer_id", "invoice_date"]))
    gaps = orders.groupby("customer_id")["invoice_date"].diff().dt.total_seconds() / 86400
    avg_gap = gaps.groupby(orders["customer_id"]).mean().reindex(idx)

    res = pd.DataFrame({
        "customer_id": idx,
        "snapshot_date": snapshot_date,
        "recency_days": (snapshot_date - out["last_date"]).dt.days.values,
        "frequency_orders": out["frequency_orders"].values,
        "monetary": out["monetary"].values,
        "average_order_value": (out["monetary"] / out["frequency_orders"]).values,
        "average_basket_quantity": (out["total_quantity"] / out["frequency_orders"]).values,
        "distinct_products": out["distinct_products"].values,
        "tenure_days": (out["last_date"] - out["first_date"]).dt.days.values,
        "average_days_between_orders": avg_gap.values,
    })
    for w in (30, 90, 180):
        sub = hist.loc[hist["invoice_date"] >= snapshot_date - pd.Timedelta(days=w)]
        res[f"orders_last_{w}d"] = per_customer(sub, "invoice", "nunique", "o", idx).astype(int).values
    for w in (30, 90, 180):
        sub = hist.loc[hist["invoice_date"] >= snapshot_date - pd.Timedelta(days=w)]
        res[f"spend_last_{w}d"] = per_customer(sub, "line_total", "sum", "s", idx).values

    c_lines = per_customer(canc, "invoice", "size", "n", idx)
    c_value = canc.assign(v=canc["line_total"].abs()).groupby("customer_id")["v"].sum().reindex(idx).fillna(0)
    c_inv = per_customer(canc, "invoice", "nunique", "i", idx)
    res["cancellation_count"] = c_lines.astype(int).values
    res["cancellation_value"] = c_value.values
    res["cancellation_rate"] = (c_inv / np.maximum(out["frequency_orders"] + c_inv, 1)).values
    res["country"] = out["country"].values
    res["lapse"] = (~idx.isin(future_buyers)).astype(int)
    snapshot_tables.append(res)

customer_snapshots = pd.concat(snapshot_tables, ignore_index=True)
customer_snapshots["customer_id"] = customer_snapshots["customer_id"].astype("string")
customer_snapshots["snapshot_date"] = pd.to_datetime(customer_snapshots["snapshot_date"])
print("Snapshot rows:", len(customer_snapshots))
print("Snapshot dates:", sorted(customer_snapshots["snapshot_date"].dt.date.unique()))
display(customer_snapshots.head())

Snapshot rows: 24866
Snapshot dates: [datetime.date(2010, 6, 1), datetime.date(2010, 9, 1), datetime.date(2010, 12, 1), datetime.date(2011, 3, 1), datetime.date(2011, 6, 1), datetime.date(2011, 9, 1)]


,customer_id,snapshot_date,recency_days,frequency_orders,monetary,average_order_value,average_basket_quantity,distinct_products,tenure_days,average_days_between_orders,orders_last_30d,orders_last_90d,orders_last_180d,spend_last_30d,spend_last_90d,spend_last_180d,cancellation_count,cancellation_value,cancellation_rate,country,lapse
0,12346,2010-06-01,90,1,27.05,27.0500,5.00,5,0,NaN,0,0,1,0.00,0.00,27.05,0,0.00,0.000000,United Kingdom,0
1,12349,2010-06-01,32,1,1068.52,1068.5200,473.00,46,0,NaN,0,1,1,0.00,1068.52,1068.52,5,24.15,0.500000,Italy,1
2,12355,2010-06-01,10,1,488.21,488.2100,303.00,22,0,NaN,1,1,1,488.21,488.21,488.21,0,0.00,0.000000,Bahrain,1
3,12358,2010-06-01,174,1,1429.83,1429.8300,309.00,17,0,NaN,0,0,1,0.00,0.00,1429.83,0,0.00,0.000000,Austria,0
4,12359,2010-06-01,69,4,1522.23,380.5575,194.25,65,107,35.93588,0,2,4,0.00,683.34,1522.23,4,94.00,0.333333,Cyprus,0


### Split lapse snapshots chronologically
- **What we are doing:** Assign snapshot rows to training, validation, and test sets by their snapshot dates, then report customer counts and lapse percentages.
- **How to read the table:** Lapse percentage is the share of rows labeled `1`.
- Compare it across splits; large differences may affect model evaluation.
- **Critical caution:** If a split is empty, the code stops. Do not quietly replace this with random splitting: first check the eligible snapshot dates and whether each date has a complete 90-day future window. - This setup uses specific dates, so the dataset's date coverage must support them.

In [19]:
train_snapshots = customer_snapshots.loc[
    customer_snapshots["snapshot_date"].isin(TRAIN_SNAPSHOT_DATES)
].copy()
validation_snapshots = customer_snapshots.loc[
    customer_snapshots["snapshot_date"] == VALIDATION_SNAPSHOT_DATE
].copy()
test_snapshots = customer_snapshots.loc[
    customer_snapshots["snapshot_date"] == TEST_SNAPSHOT_DATE
].copy()

if train_snapshots.empty or validation_snapshots.empty or test_snapshots.empty:
    raise ValueError(
        "At least one lapse snapshot split is empty. Check eligible snapshot "
        "dates and whether each has a complete 90-day future window."
    )

def snapshot_split_summary(name, data):
    return {
        "Split": name,
        "First Snapshot": data["snapshot_date"].min(),
        "Last Snapshot": data["snapshot_date"].max(),
        "Rows": len(data),
        "Unique Customers": data["customer_id"].nunique(),
        "Lapse Count": int(data["lapse"].sum()),
        "Retained Count": int((data["lapse"] == 0).sum()),
        "Lapse Percentage": data["lapse"].mean() * 100,
    }

lapse_split_summary = pd.DataFrame([
    snapshot_split_summary("Train", train_snapshots),
    snapshot_split_summary("Validation", validation_snapshots),
    snapshot_split_summary("Test", test_snapshots),
])
display(lapse_split_summary)

,Split,First Snapshot,Last Snapshot,Rows,Unique Customers,Lapse Count,Retained Count,Lapse Percentage
0,Train,2010-06-01,2011-03-01,14734,4512,8485,6249,57.587892
1,Validation,2011-06-01,2011-06-01,4908,4908,3291,1617,67.053790
2,Test,2011-09-01,2011-09-01,5224,5224,2988,2236,57.197550


### Check snapshot labels and time boundaries
- **What we are doing:** Recalculate whether each customer purchased in the 90 days after each snapshot and compare that result with the stored `lapse` label. 
- Then check that training, validation, and test snapshots are in chronological order.
- **What success means:** If the checks pass, the labels match the code's stated 90-day definition and the split dates are ordered correctly.
- **Keep in mind:** Passing these checks confirms consistency with our rules; it does not prove that the business definition of “lapse” is perfect. 
- It also does not guarantee every other feature is free of leakage, so keep features historical.

In [20]:
def check_snapshot_labels(snapshot_data):
    for snapshot_date, rows in snapshot_data.groupby("snapshot_date"):
        window_end = snapshot_date + pd.Timedelta(days=LABEL_WINDOW_DAYS)

        # Everyone who bought something in the 90 days after this snapshot
        buyers = purchase_data.loc[
            (purchase_data["invoice_date"] > snapshot_date)
            & (purchase_data["invoice_date"] <= window_end),
            "customer_id",
        ].unique()

        expected_lapse = (~rows["customer_id"].isin(buyers)).astype(int)
        wrong = rows.loc[rows["lapse"].to_numpy() != expected_lapse.to_numpy()]
        assert wrong.empty, (
            f"{len(wrong)} wrong lapse labels at {snapshot_date}, "
            f"first customer: {wrong['customer_id'].iloc[0]}"
        )

    assert (snapshot_data["snapshot_date"] <= data_end_date).all()

check_snapshot_labels(train_snapshots)
check_snapshot_labels(validation_snapshots)
check_snapshot_labels(test_snapshots)

assert train_snapshots["snapshot_date"].max() < validation_snapshots["snapshot_date"].min()
assert validation_snapshots["snapshot_date"].max() < test_snapshots["snapshot_date"].min()

# NEW: each split's 90-day label window must finish before the next split's first snapshot
label_window = pd.Timedelta(days=LABEL_WINDOW_DAYS)
assert (train_snapshots["snapshot_date"].max() + label_window
        <= validation_snapshots["snapshot_date"].min()), "Train labels overlap validation"
assert (validation_snapshots["snapshot_date"].max() + label_window
        <= test_snapshots["snapshot_date"].min()), "Validation labels overlap test"
print("Snapshot label and chronological checks passed (no label-window overlap).")

Snapshot label and chronological checks passed (no label-window overlap).


### Feature dictionary


In [21]:
feature_dictionary = pd.DataFrame([
    ["customer_id", "Customer identifier", "String", "Key, not a numeric predictor"],
    ["snapshot_date", "Date features are calculated", "Date", "Features use earlier transactions"],
    ["recency_days", "Days since last purchase", "Integer", "Lower means more recent activity"],
    ["frequency_orders", "Distinct historical purchase invoices", "Integer", "Historical order frequency"],
    ["monetary", "Sum of historical line totals", "Numeric", "Gross value, not profit"],
    ["average_order_value", "Monetary divided by invoice count", "Numeric", "Historical average"],
    ["average_basket_quantity", "Quantity divided by invoice count", "Numeric", "Historical average units"],
    ["distinct_products", "Distinct historical stock codes", "Integer", "Historical product variety"],
    ["tenure_days", "First-to-last historical purchase span", "Integer", "Limited by dataset start"],
    ["average_days_between_orders", "Mean gap between historical orders", "Numeric", "Missing for single-order customers"],
    ["orders_last_30d", "Orders in preceding 30 days", "Integer", "Snapshot-specific historical feature"],
    ["orders_last_90d", "Orders in preceding 90 days", "Integer", "Snapshot-specific historical feature"],
    ["orders_last_180d", "Orders in preceding 180 days", "Integer", "Snapshot-specific historical feature"],
    ["spend_last_30d", "Gross value in preceding 30 days", "Numeric", "Snapshot-specific historical feature"],
    ["spend_last_90d", "Gross value in preceding 90 days", "Numeric", "Snapshot-specific historical feature"],
    ["spend_last_180d", "Gross value in preceding 180 days", "Numeric", "Snapshot-specific historical feature"],
    ["cancellation_count", "Historical cancellation lines", "Integer", "Not a verified return count"],
    ["cancellation_value", "Absolute historical cancellation line value", "Numeric", "Proxy, not verified return value"],
    ["cancellation_rate", "Cancellation invoices / purchase plus cancellation invoices", "Numeric", "Historical proxy"],
    ["country", "Most recently observed country", "Categorical", "Encode inside training pipeline"],
    ["lapse", "No purchase in following 90 days", "Binary", "1 = lapse; 0 = at least one purchase"],
], columns=["Feature", "Definition", "Type", "Notes"])


display(feature_dictionary.head())

,Feature,Definition,Type,Notes
0,customer_id,Customer identifier,String,"Key, not a numeric predictor"
1,snapshot_date,Date features are calculated,Date,Features use earlier transactions
2,recency_days,Days since last purchase,Integer,Lower means more recent activity
3,frequency_orders,Distinct historical purchase invoices,Integer,Historical order frequency
4,monetary,Sum of historical line totals,Numeric,"Gross value, not profit"


### Export the core team datasets


In [22]:
drop_before_export = [
    "month", "invalid_date", "invalid_quantity", "invalid_price",
    "is_cancellation", "is_non_product", "is_purchase", "transaction_type",
    "is_negative_qty_non_cancel", "is_reversed_entry",
]

for name, part in [("train_purchases_v1.csv", train_transactions),
                   ("validation_purchases_v1.csv", validation_transactions),
                   ("test_purchases_v1.csv", test_transactions)]:
    part.drop(columns=drop_before_export, errors="ignore").to_csv(OUTPUT_DIR / name, index=False)

train_customer_product.to_csv(
    OUTPUT_DIR / "train_customer_product_interactions_v1.csv", index=False
)

train_snapshots.to_csv(OUTPUT_DIR / "lapse_train_v1.csv", index=False)
validation_snapshots.to_csv(OUTPUT_DIR / "lapse_validation_v1.csv", index=False)
test_snapshots.to_csv(OUTPUT_DIR / "lapse_test_v1.csv", index=False)
train_product_catalog.to_csv(OUTPUT_DIR / "train_product_catalog_v1.csv", index=False)

- Supporting tables for Member 4; and may be useful for cancellation-aware ranking.

In [23]:
customer_cancel_rates.to_csv(
    OUTPUT_DIR / "train_customer_cancellation_rates_v1.csv", index=False
)
product_cancel_rates.to_csv(
    OUTPUT_DIR / "train_product_cancellation_rates_v1.csv", index=False
)
cancellation_links.to_csv(
    OUTPUT_DIR / "cancellation_link_candidates_v1.csv", index=False
)

cleaning_summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False)
split_summary.to_csv(OUTPUT_DIR / "purchase_split_summary.csv", index=False)
feature_dictionary.to_csv(OUTPUT_DIR / "feature_dictionary.csv", index=False)


### Final data-quality checks
- **What we are doing:** Run assertions that stop the script if key assumptions are broken—for example, duplicate rows remain, purchase rows have invalid values, cancellation/non-product rows slipped into purchases, or time splits overlap.
- **What success means:** “All final data-quality checks passed” means these specific programmed checks passed. The final table lists the CSV files actually created.
- **Keep in mind:** A passing check does not prove that every business rule is correct. 
- It cannot prove that non-merchandise classification is perfect or that cancellation links are genuine. Review those summaries separately before teammates interpret them as labels or train models.

In [24]:
required_clean_columns = {
    "invoice", "stock_code", "invoice_date", "quantity", "price",
    "customer_id", "is_cancellation", "is_purchase", "line_total",
    "is_non_product", "transaction_type",
    "is_negative_qty_non_cancel", "is_reversed_entry",
}
assert required_clean_columns.issubset(clean.columns)
if REMOVE_EXACT_DUPLICATES:
    assert not clean.duplicated().any()
assert clean["line_total"].equals(clean["quantity"] * clean["price"])

# every row got a real category
assert not clean["transaction_type"].eq("other").any()
assert clean.loc[clean["is_purchase"], "transaction_type"].eq("purchase").all()

# purchases
assert purchases["invoice_date"].notna().all()
assert purchases["quantity"].gt(0).all()
assert purchases["price"].gt(0).all()
assert not purchases["is_cancellation"].any()
assert not purchases["is_non_product"].any()
assert not purchases["is_negative_qty_non_cancel"].any()
assert not purchases["is_reversed_entry"].any()

# cancellations
assert cancellations["is_cancellation"].all()
assert not cancellations["is_non_product"].any()
assert not cancellations["is_reversed_entry"].any()

# time splits
assert (train_transactions["invoice_date"].max()
        < validation_transactions["invoice_date"].min())
assert (validation_transactions["invoice_date"].max()
        < test_transactions["invoice_date"].min())
assert train_snapshots["snapshot_date"].max() < validation_snapshots["snapshot_date"].min()
assert validation_snapshots["snapshot_date"].max() < test_snapshots["snapshot_date"].min()
label_window = pd.Timedelta(days=LABEL_WINDOW_DAYS)
assert train_snapshots["snapshot_date"].max() + label_window <= validation_snapshots["snapshot_date"].min()
assert validation_snapshots["snapshot_date"].max() + label_window <= test_snapshots["snapshot_date"].min()

output_files = sorted(path.name for path in OUTPUT_DIR.glob("*.csv"))
print("All final data-quality checks passed.")
print("Output directory:", OUTPUT_DIR.resolve())
print("CSV files created:", len(output_files))
display(pd.DataFrame({"File": output_files}))

All final data-quality checks passed.
Output directory: C:\Users\Retaj\Downloads\capstone project 3 SIC\data engineering and EDA\retail_outputs
CSV files created: 14


,File
0,cancellation_link_candidates_v1.csv
1,cleaning_summary.csv
2,feature_dictionary.csv
3,lapse_test_v1.csv
4,lapse_train_v1.csv
5,lapse_validation_v1.csv
6,purchase_split_summary.csv
7,test_purchases_v1.csv
8,train_customer_cancellation_rates_v1.csv
9,train_customer_product_interactions_v1.csv
